In [ ]:
import json
import pickle
from dataclasses import dataclass
from copy import deepcopy
from pathlib import Path
from typing import Any, Dict, List

import pandas as pd
import numpy as np
import py3Dmol
import rdkit
from rdkit import Chem
from scipy.optimize import NonlinearConstraint, minimize

from ringer.sidechain_reconstruction import (
    Macrocycle,
    Reconstructor,
    set_rdkit_geometries,
)
from ringer.utils import reconstruction, internal_coords
from ringer.utils.internal_coords import (
    compute_canonical_transform,
    apply_affine_transform,
    angle_and_dihedral_loss,
)
from scripts.reconstruct_single import merge_samples, subset_structure_conformers

In [2]:
ROOT = Path.cwd()
if not (ROOT / "ringer").exists():
    ROOT = ROOT.parent

MOL_NAME = "F.Mec.q.Y.pickle"
MOL_IDX = 827

PRE_OPT_PATH = ROOT / "sample/reconstructed_mols_no_opt" / MOL_NAME
TEST_MOL_DIR = ROOT / "data/cremp/test"
SAMPLES_PATH = ROOT / "sample/samples.pickle"
MEAN_DIST_PATH = ROOT / "assets/models/conditional/training_mean_distances.json"

MAX_CONF = 1
MAXITER = 100

BACKBONE_ATOM_IDS = [33, 27, 25, 24, 15, 13, 12, 4, 2, 1, 36, 34]

for p in [PRE_OPT_PATH, SAMPLES_PATH, MEAN_DIST_PATH]:
    assert p.exists(), p
print(ROOT)

/mnt/HDD1/Codes/ringer


In [3]:
def extract_backbone(mol, backbone_atom_ids):
    rwmol = rdkit.Chem.RWMol(mol)
    # Remove in reverse order (indices shift otherwise)
    keep = set(backbone_atom_ids)
    for idx in sorted(range(rwmol.GetNumAtoms()), reverse=True):
        if idx not in keep:
            rwmol.RemoveAtom(idx)
    backbone_mol2 = rwmol.GetMol()
    return backbone_mol2

In [4]:
def load_pickle(path: Path):
    with path.open("rb") as f:
        return pickle.load(f)

In [5]:
def visualize_conformer(mol, title, id=0):
    viewer = py3Dmol.view(width=400, height=400)
    if mol.GetNumConformers() > 0:
        mol_block = rdkit.Chem.MolToMolBlock(mol, confId=id)
        viewer.addModel(mol_block, "mol")
        viewer.setStyle({"stick": {"color": "spectrum"}})
        viewer.zoomTo()
        viewer.setBackgroundColor("white")
        print(f"{title} - Conformer {id}")
        return viewer
    else:
        print(f"{title} has no conformers")
        return None

In [6]:
def get_structure_from_coords(structure, coords: List[pd.DataFrame]) -> Dict[str, Any]:
    # Convert list of coords to structure
    # Concatenate making hierarchical index of sample_idx and atom_idx
    coords_stacked = pd.concat(
        coords, keys=range(len(coords)), names=["sample_idx", "atom_idx"]
    )
    # Pivot so we can get all samples for each feature from the outermost column
    coords_pivoted = coords_stacked.unstack(level="atom_idx")
    new_structure = {
        feat_name: coords_pivoted[feat_name]
        for feat_name in coords_pivoted.columns.levels[0]
    }
    new_structure["atom_labels"] = structure["atom_labels"]
    return new_structure

In [7]:
structures_dict = load_pickle(SAMPLES_PATH)
fname = list(structures_dict.keys())[MOL_IDX]
assert (
    Path(fname).name == MOL_NAME
), f"Index {MOL_IDX} is {Path(fname).name}, not {MOL_NAME}"
structure = structures_dict[fname]

In [8]:
with MEAN_DIST_PATH.open("r") as f:
    mean_bond_dists = json.load(f)

ring_idxs = list(structure["dihedral"].columns)
ring_internal = internal_coords.RingInternalCoordinates(ring_idxs)

bond_dists = pd.Series(
    data=(mean_bond_dists[label] for label in structure["atom_labels"]),
    index=ring_idxs,
    dtype=float,
)

In [9]:
test_mol_path = TEST_MOL_DIR / MOL_NAME
assert test_mol_path.exists(), test_mol_path
test_mol = load_pickle(test_mol_path)
if isinstance(test_mol, dict):
    test_mol = test_mol["rd_mol"]

In [10]:
mol_A, coords_A, unsuccessful_A = reconstruction.reconstruct_ring(
    mol=deepcopy(test_mol),
    structure=structure,
    bond_dist_dict=mean_bond_dists,
    angles_as_constraints=False,
    opt_init="best_dists",
    skip_opt=True,
    drop_unsuccessful=True,
    max_conf=MAX_CONF,
    return_unsuccessful=True,
    ncpu=1,
)

  0%|          | 0/1 [00:00<?, ?it/s]

In [11]:
mol_B, coords_B, unsuccessful_B = reconstruction.reconstruct_ring(
    mol=deepcopy(test_mol),
    structure=structure,
    bond_dist_dict=mean_bond_dists,
    angles_as_constraints=False,
    opt_init="best_dists",
    skip_opt=False,
    drop_unsuccessful=True,
    max_conf=MAX_CONF,
    return_unsuccessful=True,
    ncpu=1,
)

  0%|          | 0/1 [00:00<?, ?it/s]

In [12]:
visualize_conformer(mol_A, "No opt 827", 0)

No opt 827 - Conformer 0


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [13]:
visualize_conformer(mol_B, "No opt 827", 0)

No opt 827 - Conformer 0


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [14]:
visualize_conformer(test_mol, "No opt 827", 0)

No opt 827 - Conformer 0


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [15]:
processed_conf_idxs = list(structure["dihedral"].index)
processed_conf_idxs = processed_conf_idxs[:MAX_CONF]

failed_conf_idxs = set(unsuccessful_A.keys())
successful_conf_idxs = [
    conf_idx for conf_idx in processed_conf_idxs if conf_idx not in failed_conf_idxs
]

In [16]:
structure_filtered = subset_structure_conformers(structure, successful_conf_idxs)

In [17]:
reconstruction_config = load_pickle(
    Path("ringer/sidechain_reconstruction/data/reconstruction_data.pickle")
)

In [18]:
structure_opt = get_structure_from_coords(structure, coords_A)

In [19]:
sample = merge_samples(structure_opt, structure_filtered)

In [20]:
mol_opt_no_h = rdkit.Chem.RemoveHs(mol_A)

In [21]:
mc = Macrocycle(
    mol_opt_no_h,
    reconstruction_config,
    coords=False,
    copy=True,
    verify=True,
)
reconstructor = Reconstructor(mc)

internals_tensor = reconstructor.parse_internals(sample)
index_tensor = reconstructor.stacked_tuples

positions = reconstructor.reconstruct(internals_tensor, index_tensor)
mol_opt = set_rdkit_geometries(mol_opt_no_h, positions, add_hydrogens=True, copy=True)

In [22]:
visualize_conformer(mol_opt, "No opt 827", 0)

No opt 827 - Conformer 0


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [23]:
backbone_mol1 = extract_backbone(mol_A, BACKBONE_ATOM_IDS)
backbone_mol2 = extract_backbone(mol_opt, BACKBONE_ATOM_IDS)
backbone_mol3 = extract_backbone(mol_B, BACKBONE_ATOM_IDS)

In [24]:
print(rdkit.Chem.AllChem.GetBestRMS(backbone_mol1, backbone_mol2))

0.005651216110138537


# Trajectory

In [25]:
@dataclass
class TrajectoryResult:
    conf_idx: int
    success: bool
    niter: int
    message: str
    objective: List[float]
    ring_xyz_frames: List[np.ndarray]
    final_xyz: np.ndarray

In [26]:
def optimize_from_premol(
    pre_mol: rdkit.Chem.Mol,
    conf_idx: int,
) -> TrajectoryResult:
    angle_target = structure["angle"].loc[conf_idx]
    dihedral_target = structure["dihedral"].loc[conf_idx]

    xyz0_raw = pre_mol.GetConformer(conf_idx).GetPositions()[ring_idxs]
    trans_mat = compute_canonical_transform(xyz0_raw)
    inv_trans = np.linalg.inv(trans_mat)
    x0 = apply_affine_transform(xyz0_raw, trans_mat).ravel()

    objective: List[float] = []
    x_frames: List[np.ndarray] = [x0.copy()]

    def objective_with_grad(x: np.ndarray):
        loss, grad = angle_and_dihedral_loss(
            x,
            internal_coords=ring_internal,
            angle_vals_target=np.asarray(angle_target),
            dihedral_vals_target=np.asarray(dihedral_target),
            grad=True,
        )
        objective.append(float(loss))
        return loss, grad

    dist_constraint = NonlinearConstraint(
        ring_internal.compute_distances,
        lb=np.asarray(bond_dists),
        ub=np.asarray(bond_dists),
        jac=ring_internal.compute_distance_jacobian,
    )

    def callback(xk: np.ndarray, *args):
        x_frames.append(np.asarray(xk).copy())

    result = minimize(
        objective_with_grad,
        x0,
        constraints=[dist_constraint],
        jac=True,
        options={"maxiter": MAXITER},
        callback=callback,
    )

    if not np.allclose(x_frames[-1], result.x):
        x_frames.append(np.asarray(result.x).copy())

    ring_xyz_frames = [
        apply_affine_transform(x.reshape(-1, 3), inv_trans) for x in x_frames
    ]

    return TrajectoryResult(
        conf_idx=int(conf_idx),
        success=bool(result.success),
        niter=int(getattr(result, "nit", len(x_frames) - 1)),
        message=str(result.message),
        objective=objective,
        ring_xyz_frames=ring_xyz_frames,
        final_xyz=ring_xyz_frames[-1],
    )

In [27]:
results_A: Dict[int, TrajectoryResult] = {}
for conf_idx in range(MAX_CONF):
    results_A[conf_idx] = optimize_from_premol(mol_A, conf_idx)

In [28]:
def _mol_with_replaced_ring(
    mol_template: Chem.Mol, ring_idxs: List[int], ring_xyz: np.ndarray
) -> Chem.Mol:
    conf = Chem.Conformer(mol_template.GetConformer(0))
    for atom_idx, pos in zip(ring_idxs, np.asarray(ring_xyz, dtype=float)):
        conf.SetAtomPosition(
            int(atom_idx), [float(pos[0]), float(pos[1]), float(pos[2])]
        )
    new_mol = Chem.Mol(mol_template, quickCopy=True)
    new_mol.RemoveAllConformers()
    new_mol.AddConformer(conf, assignId=True)
    return extract_backbone(new_mol, BACKBONE_ATOM_IDS)


def export_trajectory_sdf(
    base_mol: Chem.Mol,
    traj: TrajectoryResult,
    ring_idxs: List[int],
    out_path: Path,
    method: str = "",
) -> Path:
    mol_template = Chem.Mol(base_mol, quickCopy=True)
    mol_template.RemoveAllConformers()
    mol_template.AddConformer(Chem.Conformer(base_mol.GetConformer(0)), assignId=True)

    writer = Chem.SDWriter(str(out_path))
    for step_idx, ring_xyz in enumerate(traj.ring_xyz_frames):
        mol_step = _mol_with_replaced_ring(mol_template, ring_idxs, ring_xyz)
        mol_step.SetProp("conf_idx", str(traj.conf_idx))
        mol_step.SetProp("opt_step", str(step_idx))
        mol_step.SetProp("n_frames", str(len(traj.ring_xyz_frames)))
        mol_step.SetProp("method", method)
        writer.write(mol_step)
    writer.close()
    return out_path


OUT_DIR = ROOT / "sample/analysis/optimization_trajectory/"
OUT_DIR.mkdir(parents=True, exist_ok=True)

for conf_idx in range(MAX_CONF):
    export_trajectory_sdf(
        base_mol=mol_A,
        traj=results_A[conf_idx],
        ring_idxs=ring_idxs,
        out_path=OUT_DIR / f"{MOL_NAME}_conf{conf_idx}_A_premol.sdf",
        method="A_premol",
    )